In [1]:
import os
import math
import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

load_path = "/home/sz4544/EEG-motor-imagery-main/project/train1800_raw_EEG.h5"
model_path = "/home/sz4544/EEG-motor-imagery-main/project/models/diffusion_train1800_v2.pt"
scaler_path = "/home/sz4544/EEG-motor-imagery-main/project/models/diffusion_train1800_v2_scaler.npz"

os.makedirs("/home/sz4544/EEG-motor-imagery-main/project/models", exist_ok=True)


f = h5py.File(load_path, "r")
x_train = f["data"][:]
y_train = f["tasks"][:]
f.close()

print(x_train.shape, y_train.shape)
print(np.unique(y_train))


y_train = y_train.astype(np.int64)
if np.array_equal(np.unique(y_train), np.array([1, 2, 3, 4])):
    y_train = y_train - 1

x_train = x_train.astype(np.float32)

mean = x_train.mean(axis=(0, 1), keepdims=True)
std = x_train.std(axis=(0, 1), keepdims=True) + 1e-6

x_train = (x_train - mean) / std

np.savez(scaler_path, mean=mean, std=std)

print(x_train.shape, y_train.shape)
print(np.unique(y_train))
print("standardized train mean:", x_train.mean())
print("standardized train std:", x_train.std())


x_train = np.transpose(x_train, (0, 2, 1))  # (N, 64, 640)
print(x_train.shape)

class EEGDiffusionDataset(Dataset):
    def __init__(self, x, y):
        self.x = torch.tensor(x, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

dataset = EEGDiffusionDataset(x_train, y_train)
loader = DataLoader(dataset, batch_size=32, shuffle=True, drop_last=True)

print(len(dataset), len(loader))

class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half_dim = self.dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=t.device) * -emb)
        emb = t[:, None] * emb[None, :]
        emb = torch.cat((emb.sin(), emb.cos()), dim=1)
        return emb

class Block(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim, class_dim):
        super().__init__()
        self.conv1 = nn.Conv1d(in_ch, out_ch, 3, padding=1)
        self.conv2 = nn.Conv1d(out_ch, out_ch, 3, padding=1)
        self.time_mlp = nn.Linear(time_dim, out_ch)
        self.class_mlp = nn.Linear(class_dim, out_ch)
        self.bn1 = nn.BatchNorm1d(out_ch)
        self.bn2 = nn.BatchNorm1d(out_ch)

    def forward(self, x, t_emb, c_emb):
        h = self.conv1(x)
        h = h + self.time_mlp(t_emb).unsqueeze(-1)
        h = h + self.class_mlp(c_emb).unsqueeze(-1)
        h = F.silu(self.bn1(h))
        h = self.conv2(h)
        h = F.silu(self.bn2(h))
        return h

class SimpleCondUNet1D(nn.Module):
    def __init__(self, n_channels=64, n_classes=4, base=64, time_dim=128, class_dim=64):
        super().__init__()
        self.time_emb = SinusoidalPosEmb(time_dim)
        self.class_emb = nn.Embedding(n_classes, class_dim)

        self.in_proj = nn.Conv1d(n_channels, base, 3, padding=1)
        self.block1 = Block(base, base, time_dim, class_dim)
        self.down1 = nn.Conv1d(base, base * 2, 4, stride=2, padding=1)
        self.block2 = Block(base * 2, base * 2, time_dim, class_dim)
        self.down2 = nn.Conv1d(base * 2, base * 4, 4, stride=2, padding=1)
        self.block3 = Block(base * 4, base * 4, time_dim, class_dim)

        self.up1 = nn.ConvTranspose1d(base * 4, base * 2, 4, stride=2, padding=1)
        self.block4 = Block(base * 4, base * 2, time_dim, class_dim)
        self.up2 = nn.ConvTranspose1d(base * 2, base, 4, stride=2, padding=1)
        self.block5 = Block(base * 2, base, time_dim, class_dim)

        self.out_proj = nn.Conv1d(base, n_channels, 3, padding=1)

    def forward(self, x, t, y):
        t_emb = self.time_emb(t)
        c_emb = self.class_emb(y)

        x0 = self.in_proj(x)
        x1 = self.block1(x0, t_emb, c_emb)
        x2 = self.down1(x1)
        x2 = self.block2(x2, t_emb, c_emb)
        x3 = self.down2(x2)
        x3 = self.block3(x3, t_emb, c_emb)

        x = self.up1(x3)
        x = torch.cat([x, x2], dim=1)
        x = self.block4(x, t_emb, c_emb)

        x = self.up2(x)
        x = torch.cat([x, x1], dim=1)
        x = self.block5(x, t_emb, c_emb)

        return self.out_proj(x)


T = 1000
betas = torch.linspace(1e-4, 0.02, T).to(device)
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)

def q_sample(x0, t, noise):
    a = alphas_cumprod[t].view(-1, 1, 1)
    return torch.sqrt(a) * x0 + torch.sqrt(1 - a) * noise


model = SimpleCondUNet1D().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

epochs = 50
model.train()

for epoch in range(epochs):
    losses = []
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        t = torch.randint(0, T, (xb.size(0),), device=device)
        noise = torch.randn_like(xb)
        xt = q_sample(xb, t, noise)

        pred = model(xt, t.float(), yb)
        loss = F.mse_loss(pred, noise)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

    print(f"epoch {epoch+1}/{epochs} loss={np.mean(losses):.6f}")

torch.save(model.state_dict(), model_path)
print("saved:", model_path)




cuda
(1800, 640, 64) (1800,)
[1. 2. 3. 4.]
(1800, 640, 64) (1800,)
[0 1 2 3]
standardized train mean: -2.8500283e-06
standardized train std: 0.9842581
(1800, 64, 640)
1800 56
epoch 1/50 loss=1.048777
epoch 2/50 loss=1.001557
epoch 3/50 loss=0.989078
epoch 4/50 loss=0.968346
epoch 5/50 loss=0.940297
epoch 6/50 loss=0.912929
epoch 7/50 loss=0.888202
epoch 8/50 loss=0.864434
epoch 9/50 loss=0.840930
epoch 10/50 loss=0.818015
epoch 11/50 loss=0.798993
epoch 12/50 loss=0.779245
epoch 13/50 loss=0.764689
epoch 14/50 loss=0.753055
epoch 15/50 loss=0.738413
epoch 16/50 loss=0.730746
epoch 17/50 loss=0.714046
epoch 18/50 loss=0.705498
epoch 19/50 loss=0.695196
epoch 20/50 loss=0.678752
epoch 21/50 loss=0.672988
epoch 22/50 loss=0.664295
epoch 23/50 loss=0.654982
epoch 24/50 loss=0.650145
epoch 25/50 loss=0.646842
epoch 26/50 loss=0.637592
epoch 27/50 loss=0.626914
epoch 28/50 loss=0.624041
epoch 29/50 loss=0.616767
epoch 30/50 loss=0.607848
epoch 31/50 loss=0.603029
epoch 32/50 loss=0.598529
ep